# 📓 Semana 4 · Dia 4 — Camada Prata: dimensões e tabela fato (Star Schema)

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEA (medallion) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Prata completa: dim_cliente, dim_produto, dim_tempo, fato_vendas |

---


## 📖 Teoria — O papel da Prata

A camada Prata transforma o Bronze cru em um modelo limpo e reutilizável: deduplicado, tipado, validado e modelado (dimensões + fatos). Regras:
- **Idempotente**: rodar N vezes = mesmo resultado.
- **Sem dados brutos**: só o que é necessário.
- **Chaves surrogate** (`sk_*`) para cada dimensão.


### 💻 Na prática — Criando as dimensões

Comece pelas dimensões — elas são as 'tabelas de contexto'.


In [ ]:
# dim_cliente: deduplicado e tipado
from pyspark.sql.functions import col, row_number, min, max, count, sum as s
from pyspark.sql.window import Window
df = spark.table("workspace.bronze.vendas_bronze")
clientes = (df
    .filter(col("CustomerID").isNotNull())
    .groupBy("CustomerID", "Country")
    .agg(count("*").alias("n_vendas"),
         min("InvoiceDate").alias("primeira_compra"),
         max("InvoiceDate").alias("ultima_compra"))
    .withColumn("sk_cliente", row_number().over(Window.orderBy("CustomerID"))))
clientes.createOrReplaceTempView("dim_cliente_vw")
clientes.show(5, truncate=False)
print("Total de clientes:", clientes.count())

In [ ]:
# dim_produto
produtos = (df
    .select("StockCode", "Description")
    .dropDuplicates(["StockCode"])
    .filter(col("StockCode").isNotNull())
    .withColumn("sk_produto", row_number().over(Window.orderBy("StockCode"))))
produtos.createOrReplaceTempView("dim_produto_vw")
produtos.show(5, truncate=False)

In [ ]:
# dim_tempo (a partir das datas das vendas)
from pyspark.sql.functions import to_date, year, month, dayofmonth, quarter
datas = (df
    .select(to_date("InvoiceDate", "M/d/yyyy H:mm").alias("data_venda"))
    .dropDuplicates()
    .filter(col("data_venda").isNotNull())
    .withColumn("ano", year("data_venda"))
    .withColumn("mes", month("data_venda"))
    .withColumn("dia", dayofmonth("data_venda"))
    .withColumn("trimestre", quarter("data_venda"))
    .withColumn("sk_tempo", row_number().over(Window.orderBy("data_venda"))))
datas.createOrReplaceTempView("dim_tempo_vw")
datas.show(5)

### 💻 Na prática — Criando o fato

O fato conecta as dimensões por chave e carrega as medidas.


In [ ]:
# fato_vendas: junta as chaves das dimensões
from pyspark.sql.functions import to_date, col as c
fato = (df
    .filter(c("CustomerID").isNotNull())
    .withColumn("data_venda", to_date("InvoiceDate", "M/d/yyyy H:mm"))
    .join(clientes.select("CustomerID", "sk_cliente"), "CustomerID", "left")
    .join(produtos.select("StockCode", "sk_produto"), "StockCode", "left")
    .join(datas.select("data_venda", "sk_tempo"), "data_venda", "left")
    .select("InvoiceNo", "sk_cliente", "sk_produto", "sk_tempo",
            "Quantity", "UnitPrice", "Country")
    .withColumn("receita", c("Quantity") * c("UnitPrice")))
fato.createOrReplaceTempView("fato_vendas_vw")
fato.show(5, truncate=False)

In [ ]:
# Gravar a Prata (idempotente)
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.prata")
clientes.write.mode("overwrite").saveAsTable("workspace.prata.dim_cliente")
produtos.write.mode("overwrite").saveAsTable("workspace.prata.dim_produto")
datas.write.mode("overwrite").saveAsTable("workspace.prata.dim_tempo")
fato.write.mode("overwrite").saveAsTable("workspace.prata.fato_vendas")
print("Prata completa!")

In [ ]:
%sql
-- Conferência: fato + dimensões via join (o star schema funcionando)
SELECT c.sk_cliente, p.sk_produto, t.ano, f.receita
FROM workspace.prata.fato_vendas f
JOIN workspace.prata.dim_cliente c ON f.sk_cliente = c.sk_cliente
JOIN workspace.prata.dim_produto p ON f.sk_produto = p.sk_produto
JOIN workspace.prata.dim_tempo t ON f.sk_tempo = t.sk_tempo
LIMIT 10

> 🎯 **Dica de prova**: A DEA cobra a ordem das camadas e as regras: Bronze append-only, Prata idempotente/limpa, Ouro denormalizado/agregado. Memorize as **regras da Medallion**.


## 🎯 Exercícios de fixação

**1.** Por que a Prata deve ser idempotente?

**2.** Qual a diferença entre sk_ (surrogate) e chave natural?

**3.** Refaça a dim_cliente adicionando a cidade do cliente como atributo.


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Idempotente

Para permitir reprocessamento sem duplicar dados: rodar 2x produz o mesmo resultado (overwrite/merge). Sem isso, pipelines falham e duplicam.

**2.** Surrogate vs natural

Surrogate (sk_) é artificial, estável e não depende da fonte (sobrevive a mudanças de chave natural). Natural é o ID do sistema de origem.

**3.** Cidade

Derive de `Country`/endereço na fonte ou de um lookup; na dim_cliente o ideal é a cidade do cadastro — para nosso dataset, podemos usar o país como proxy.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*